# 01 — InSDN Dataset Leakage Audit, Deduplication & SDN Observability Tiers

**Project:** Live SDN Controller Bottleneck & Lightweight AI Defense (KU DSAI Lab)  
**Design Policy (`DEC-009`):** Core data-loading and auditing logic lives in tested `.py` modules (`src/data/insdn_loader.py`); this notebook provides interactive visual inspection and exports `figures/fig01_insdn_leakage_and_tiers.png` directly from saved manifest/data files.


In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from data.insdn_loader import load_clean_insdn

sns.set_theme(style="whitegrid", context="notebook")
FIG_DIR = REPO_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

manifest = json.loads((REPO_ROOT / "data" / "manifests" / "insdn_manifest.json").read_text(encoding="utf-8"))
audit = manifest["audit"]
print(f"Dataset: {manifest['dataset_name']} | SHA-256: {manifest['sha256'][:16]}... | Rows: {audit['n_rows']:,} | Cols: {audit['n_columns']}")


Dataset: InSDN | SHA-256: a93d51e1c2debc72... | Rows: 343,889 | Cols: 84


## 1. Automated Leakage & Data-Quality Audit Summary
Verify four classic sources of inflated offline accuracy in `InSDN` (mirroring findings by Engelen et al. 2021 & Lanvin et al. 2023):
1. **Exact & Feature-Only Duplicates:** Identical feature rows shared across train/test splits if not deduplicated first.
2. **IP Shortcut Leakage:** Attack traffic originating from IPs that never appear in benign captures.
3. **Destination Port Concentration:** Lab service-port bias (`53/443` benign vs `0/80` attack).
4. **Dead Zero-Variance Features:** Columns carrying zero information.


In [2]:
audit_table = pd.DataFrame([
    {"Check": "Total raw rows", "Value": f"{audit['n_rows']:,}"},
    {"Check": "Normal / Attack raw split", "Value": f"{audit['binary_distribution']['Normal']:,} / {audit['binary_distribution']['Attack']:,}"},
    {"Check": "Feature-only duplicates (no IDs)", "Value": f"{audit['feature_only_duplicates_no_id']:,} ({100*audit['feature_only_duplicates_no_id']/audit['n_rows']:.2f}%)"},
    {"Check": "Zero-variance numeric columns", "Value": str(len(audit['zero_variance_columns']))},
    {"Check": "Attack rows from attack-only Src IP", "Value": f"{audit['ip_shortcut_leakage_audit']['Src IP']['attack_rows_with_attack_only_ip_pct']:.2f}%"},
    {"Check": "Attack rows targeting attack-only Dst IP", "Value": f"{audit['ip_shortcut_leakage_audit']['Dst IP']['attack_rows_with_attack_only_ip_pct']:.2f}%"},
    {"Check": "Clean non-constant numeric features (no Dst Port)", "Value": str(audit['observability_tiers']['T3_clean_numeric_count'])},
])
audit_table


,Check,Value
0,Total raw rows,"343,889"
1,Normal / Attack raw split,"68,424 / 275,465"
2,Feature-only duplicates (no IDs),"125,252 (36.42%)"
3,Zero-variance numeric columns,12
4,Attack rows from attack-only Src IP,86.44%
5,Attack rows targeting attack-only Dst IP,60.65%
6,Clean non-constant numeric features (no Dst Port),65


## 2. Visualize Dataset Deduplication, Port Bias & SDN Observability Tiers


In [3]:
X_clean, y_bin, _, tiers = load_clean_insdn(
    REPO_ROOT / "data" / "raw" / "insdn" / "Dataset.csv",
    include_dst_port=False,
    deduplicate_features=True,
)
print(f"After leakage removal & feature deduplication: {len(X_clean):,} unique rows x {X_clean.shape[1]} features")
print(f"  Normal: {(y_bin == 0).sum():,} ({100*(y_bin == 0).mean():.1f}%) | Attack: {(y_bin == 1).sum():,} ({100*(y_bin == 1).mean():.1f}%)")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Panel A: Raw vs Deduplicated class counts
raw_norm = audit["binary_distribution"]["Normal"]
raw_atk = audit["binary_distribution"]["Attack"]
dedup_norm = int((y_bin == 0).sum())
dedup_atk = int((y_bin == 1).sum())
x = np.arange(2)
w = 0.35
axes[0].bar(x - w/2, [raw_norm/1e3, raw_atk/1e3], width=w, label="Raw (343.9k)", color="#64748b")
axes[0].bar(x + w/2, [dedup_norm/1e3, dedup_atk/1e3], width=w, label=f"Deduplicated ({len(X_clean)/1e3:.1f}k)", color="#0284c7")
axes[0].set_xticks(x)
axes[0].set_xticklabels(["Normal", "Attack"])
axes[0].set_ylabel("Flow Count (Thousands)")
axes[0].set_title("(a) Feature-Duplicate Inflation in InSDN")
axes[0].legend()

# Panel B: Top Dst Port shortcut concentration
norm_ports = audit["dst_port_audit"]["top5_normal_dst_ports"]
atk_ports = audit["dst_port_audit"]["top5_attack_dst_ports"]
ports = ["0", "53", "80", "443"]
norm_vals = [100 * norm_ports.get(p, 0) / raw_norm for p in ports]
atk_vals = [100 * atk_ports.get(p, 0) / raw_atk for p in ports]
xp = np.arange(len(ports))
axes[1].bar(xp - w/2, norm_vals, width=w, label="Normal %", color="#16a34a")
axes[1].bar(xp + w/2, atk_vals, width=w, label="Attack %", color="#dc2626")
axes[1].set_xticks(xp)
axes[1].set_xticklabels([f"Port {p}" for p in ports])
axes[1].set_ylabel("Share of Class Flows (%)")
axes[1].set_title("(b) Lab Dst-Port Shortcut Bias")
axes[1].legend()

# Panel C: Feature count per SDN observability tier
tier_names = ["T1a\nUnidir OF", "T1b\nBidir OF", "T2\nHdr+Flags", "T3\nFull CIC"]
tier_counts = [len(tiers[k]) for k in ["T1a_Unidirectional_OF", "T1b_Bidirectional_OF", "T2_Header_Flags", "T3_Full_CICFlowMeter"]]
bars = axes[2].bar(tier_names, tier_counts, color=["#0ea5e9", "#2563eb", "#7c3aed", "#be123c"])
for b, c in zip(bars, tier_counts):
    axes[2].text(b.get_x() + b.get_width()/2, c + 1.5, str(c), ha="center", va="bottom", fontweight="bold")
axes[2].set_ylabel("Feature Count")
axes[2].set_ylim(0, max(tier_counts) + 12)
axes[2].set_title("(c) Features by SDN Observability Tier")

plt.tight_layout()
out_fig = FIG_DIR / "fig01_insdn_leakage_and_tiers.png"
plt.savefig(out_fig, dpi=220, bbox_inches="tight")
print(f"Saved figure -> {out_fig}")
plt.show()


After leakage removal & feature deduplication: 184,746 unique rows x 65 features
  Normal: 64,114 (34.7%) | Attack: 120,632 (65.3%)


Saved figure -> C:\Users\sijan\Desktop\intern\research\figures\fig01_insdn_leakage_and_tiers.png


<Figure size 1500x450 with 3 Axes>

## 3. Exact Feature Lists per SDN Observability Tier


In [4]:
for tname, cols in tiers.items():
    print(f"=== {tname} ({len(cols)} features) ===")
    print(", ".join(cols))
    print()


=== T1a_Unidirectional_OF (6 features) ===
Protocol, Flow Duration, Tot Fwd Pkts, TotLen Fwd Pkts, Fwd Pkts/s, Fwd Pkt Len Mean

=== T1b_Bidirectional_OF (12 features) ===
Protocol, Flow Duration, Tot Fwd Pkts, Tot Bwd Pkts, TotLen Fwd Pkts, TotLen Bwd Pkts, Flow Byts/s, Flow Pkts/s, Fwd Pkts/s, Bwd Pkts/s, Pkt Size Avg, Down/Up Ratio

=== T2_Header_Flags (25 features) ===
Protocol, Flow Duration, Tot Fwd Pkts, Tot Bwd Pkts, TotLen Fwd Pkts, TotLen Bwd Pkts, Flow Byts/s, Flow Pkts/s, Fwd Pkts/s, Bwd Pkts/s, Pkt Size Avg, Down/Up Ratio, Fwd Header Len, Bwd Header Len, FIN Flag Cnt, SYN Flag Cnt, RST Flag Cnt, PSH Flag Cnt, ACK Flag Cnt, URG Flag Cnt, Init Bwd Win Byts, Fwd Pkt Len Max, Fwd Pkt Len Min, Bwd Pkt Len Max, Bwd Pkt Len Min

=== T3_Full_CICFlowMeter (65 features) ===
Protocol, Flow Duration, Tot Fwd Pkts, Tot Bwd Pkts, TotLen Fwd Pkts, TotLen Bwd Pkts, Fwd Pkt Len Max, Fwd Pkt Len Min, Fwd Pkt Len Mean, Fwd Pkt Len Std, Bwd Pkt Len Max, Bwd Pkt Len Min, Bwd Pkt Len Mean, Bwd 